# 07 — Boundary Layer Suppression: Wind Speed & Temperature Depression

## Hypothesis

Heavy smoke aerosol loads can **suppress vertical mixing** in the atmospheric boundary layer.
When radiation is reduced (notebook 06), the surface heats less, the temperature gradient
weakens, and convective mixing slows — producing:

1. **Lower wind speeds** during and after smoke events
2. **Cooler daytime temperatures** (consistent with radiation notebook)
3. A **self-reinforcing feedback**: stagnant air traps more smoke, which suppresses more mixing

This mechanism is well documented at the regional scale (Zhao et al. 2017; Lu et al. 2025)
and is physically plausible at a single station during a major smoke event.

**Confounding caveat:** stagnant synoptic weather patterns (high pressure, light winds)
also favor both smoke accumulation and low wind speeds independently. We attempt to
address this by controlling for pressure and examining the diurnal wind cycle.

## Analyses
1. Time series: hourly PM2.5 and wind speed (2020)
2. Wind speed distributions: smoke vs clean hours (both events)
3. Diurnal wind cycle: smoke days vs clean days (2020)
4. Temperature depression: smoke vs clean hours
5. Lagged correlation: does PM2.5 *precede* wind reduction?
6. Controlling for pressure: partial correlation

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

SMOKE_THRESHOLD = 35
COLOR_2020  = '#d62728'
COLOR_2022  = '#1f77b4'
COLOR_SMOKE = '#d62728'
COLOR_CLEAN = '#2ca02c'

## 0. Load data

In [ ]:
df = pd.read_csv('../data/processed/analysis_data.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
df['is_smoke'] = df['pm2.5_lrapa'] >= SMOKE_THRESHOLD
df['date']     = df['timestamp'].dt.date

df2020 = df[df['event'] == 'Holiday Farm Fire 2020'].copy()
df2022 = df[df['event'] == 'Cedar Creek Fire 2022'].copy()

# Tag hours by day type (smoke day = any hour on a day with max PM ≥ threshold)
smoke_dates_2020 = set(
    df2020.groupby('date')['pm2.5_lrapa'].max()
    .loc[lambda x: x >= SMOKE_THRESHOLD].index
)
df2020['day_type'] = df2020['date'].apply(
    lambda d: 'Smoke day' if d in smoke_dates_2020 else 'Clean day'
)

print(f'2020: {len(df2020):,} hours  smoke hours: {df2020["is_smoke"].sum()}')
print(f'2022: {len(df2022):,} hours  smoke hours: {df2022["is_smoke"].sum()}')

## 1. Time series: PM2.5 and wind speed (2020)

Visual alignment of smoke onset with wind speed changes.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax = axes[0]
ax.fill_between(df2020['timestamp'], df2020['pm2.5_lrapa'], alpha=0.3, color=COLOR_2020)
ax.plot(df2020['timestamp'], df2020['pm2.5_lrapa'], color=COLOR_2020, linewidth=0.8)
ax.axhline(SMOKE_THRESHOLD, color='orange', linestyle='--', linewidth=1)
ax.set_ylabel('PM2.5 LRAPA (µg/m³)')
ax.set_title('Holiday Farm Fire 2020 — PM2.5 and Wind Speed', fontsize=12)

ax = axes[1]
ax.fill_between(df2020['timestamp'], df2020['wind_speed_mph'], alpha=0.3, color='steelblue')
ax.plot(df2020['timestamp'], df2020['wind_speed_mph'], color='steelblue', linewidth=0.8)
ax.set_ylabel('Wind speed (mph)')

for ax in axes:
    ax.axvspan(pd.Timestamp('2020-09-07'), pd.Timestamp('2020-09-14'),
               alpha=0.1, color=COLOR_2020)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/fig_bl_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Wind speed distributions: smoke vs clean hours

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (ev_name, ev_df, color) in zip(axes, [
    ('Holiday Farm 2020', df2020, COLOR_2020),
    ('Cedar Creek 2022',  df2022, COLOR_2022),
]):
    smoke_w = ev_df[ev_df['is_smoke']]['wind_speed_mph'].dropna()
    clean_w = ev_df[~ev_df['is_smoke']]['wind_speed_mph'].dropna()
    t, p = stats.ttest_ind(smoke_w, clean_w, equal_var=False)
    d = (smoke_w.mean() - clean_w.mean()) / np.sqrt(
        (smoke_w.std()**2 + clean_w.std()**2) / 2
    )

    ax.hist(clean_w, bins=30, alpha=0.5, color=COLOR_CLEAN, label=f'Clean (n={len(clean_w):,})', density=True)
    ax.hist(smoke_w, bins=30, alpha=0.6, color=color,       label=f'Smoke (n={len(smoke_w):,})', density=True)
    ax.axvline(clean_w.mean(), color=COLOR_CLEAN, linestyle='--', linewidth=2)
    ax.axvline(smoke_w.mean(), color=color,       linestyle='--', linewidth=2)
    ax.set_xlabel('Wind speed (mph)')
    ax.set_ylabel('Density')
    ax.set_title(
        f'{ev_name}\n'
        f'Clean={clean_w.mean():.1f}mph  Smoke={smoke_w.mean():.1f}mph  '
        f'Δ={smoke_w.mean()-clean_w.mean():+.1f}\n'
        f'p={p:.4f}  Cohen\'s d={d:.2f}',
        fontsize=10
    )
    ax.legend(fontsize=9)

plt.suptitle('Wind Speed: Smoke vs Clean Hours', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/fig_bl_wind_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Diurnal wind cycle: smoke days vs clean days (2020)

Boundary layer mixing has a strong diurnal cycle — winds typically increase in the afternoon
as the boundary layer deepens. If smoke suppresses mixing, we expect the **afternoon wind
increase to be blunted** on smoke days.

In [ ]:
hourly_wind = df2020.groupby(['hour', 'day_type'])['wind_speed_mph'].agg(['mean','sem']).reset_index()

fig, ax = plt.subplots(figsize=(11, 5))

for day_type, color, ls in [
    ('Smoke day', COLOR_SMOKE, '-'),
    ('Clean day', COLOR_CLEAN, '--'),
]:
    sub = hourly_wind[hourly_wind['day_type'] == day_type]
    ax.fill_between(sub['hour'],
                    sub['mean'] - sub['sem'],
                    sub['mean'] + sub['sem'],
                    alpha=0.2, color=color)
    ax.plot(sub['hour'], sub['mean'], color=color, linewidth=2.5,
            linestyle=ls, label=day_type)

ax.axvspan(10, 17, alpha=0.07, color='yellow', label='Typical afternoon mixing window')
ax.set_xlabel('Hour of day (Pacific time)')
ax.set_ylabel('Mean wind speed (mph)')
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=10)
ax.set_title(
    'Holiday Farm Fire 2020 — Diurnal wind cycle\n'
    'Smoke days vs clean days (mean ± SE)',
    fontsize=12
)
plt.tight_layout()
plt.savefig('../data/processed/fig_bl_diurnal_wind.png', dpi=150, bbox_inches='tight')
plt.show()

# Afternoon peak wind
for day_type in ['Smoke day', 'Clean day']:
    sub = df2020[df2020['day_type'] == day_type]
    aft = sub[sub['hour'].between(12, 17)]['wind_speed_mph'].mean()
    ngt = sub[sub['hour'].between(0, 5)]['wind_speed_mph'].mean()
    print(f'{day_type}: afternoon={aft:.1f} mph  overnight={ngt:.1f} mph  '
          f'afternoon-overnight={aft-ngt:+.1f}')

## 4. Temperature depression on smoke hours (2020)

Complements notebook 06 — hourly granularity rather than daily DTR.

In [ ]:
smoke_t = df2020[df2020['is_smoke']]['temperature_f'].dropna()
clean_t = df2020[~df2020['is_smoke']]['temperature_f'].dropna()
t, p = stats.ttest_ind(smoke_t, clean_t, equal_var=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(clean_t, bins=40, alpha=0.5, color=COLOR_CLEAN, density=True,
        label=f'Clean hours  mean={clean_t.mean():.1f}°F  n={len(clean_t):,}')
ax.hist(smoke_t, bins=30, alpha=0.6, color=COLOR_SMOKE, density=True,
        label=f'Smoke hours  mean={smoke_t.mean():.1f}°F  n={len(smoke_t):,}')
ax.axvline(clean_t.mean(), color=COLOR_CLEAN, linestyle='--', linewidth=2)
ax.axvline(smoke_t.mean(), color=COLOR_SMOKE, linestyle='--', linewidth=2)
ax.set_xlabel('Temperature (°F)')
ax.set_ylabel('Density')
ax.set_title(
    f'Holiday Farm Fire 2020 — Temperature during smoke vs clean hours\n'
    f'Δ = {smoke_t.mean()-clean_t.mean():+.1f}°F   p = {p:.4f}',
    fontsize=12
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../data/processed/fig_bl_temp_depression.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Lagged correlation: does PM2.5 *precede* wind reduction?

Test Pearson r between PM2.5 at lag h and wind speed at current hour
for lags 0–12 h. A negative correlation that **peaks at a positive lag**
(i.e. PM2.5 leading wind reduction) would support the directional hypothesis.

In [ ]:
lags = list(range(0, 13))
rs, ps = [], []

for lag in lags:
    pm_lagged = df2020['pm2.5_lrapa'].shift(lag)
    wind      = df2020['wind_speed_mph']
    clean     = pd.concat([pm_lagged, wind], axis=1).dropna()
    r, p = stats.pearsonr(clean.iloc[:, 0], clean.iloc[:, 1])
    rs.append(r)
    ps.append(p)

fig, ax = plt.subplots(figsize=(9, 4))
colors = [COLOR_2020 if p < 0.05 else 'gray' for p in ps]
ax.bar(lags, rs, color=colors, alpha=0.8, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Lag (hours): PM2.5 leads wind by this many hours')
ax.set_ylabel('Pearson r (PM2.5 lag → wind speed)')
ax.set_xticks(lags)
sig_patch   = mpatches.Patch(color=COLOR_2020, alpha=0.8, label='p < 0.05')
insig_patch = mpatches.Patch(color='gray',     alpha=0.8, label='p ≥ 0.05')
ax.legend(handles=[sig_patch, insig_patch], fontsize=9)
ax.set_title('Holiday Farm Fire 2020\nLagged correlation: PM2.5 → wind speed', fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/fig_bl_lag_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Lag  r       p')
for lag, r, p in zip(lags, rs, ps):
    print(f'  {lag:2d}h  {r:+.3f}  {p:.4f} {"*" if p < 0.05 else ""}')

## 6. Partial correlation: PM2.5 vs wind speed controlling for pressure

Pressure is a major confounder: synoptic high pressure → light winds AND smoke accumulation.
Partial correlation removes the linear effect of pressure from both PM2.5 and wind speed,
leaving only the variance that cannot be explained by pressure alone.

In [ ]:
from sklearn.linear_model import LinearRegression

clean = df2020[['pm2.5_lrapa', 'wind_speed_mph', 'pressure_hpa']].dropna()

def residualize(y, X):
    reg = LinearRegression().fit(X, y)
    return y - reg.predict(X)

P = clean[['pressure_hpa']].values
pm_resid   = residualize(clean['pm2.5_lrapa'].values,    P)
wind_resid = residualize(clean['wind_speed_mph'].values,  P)

r_raw,     p_raw     = stats.pearsonr(clean['pm2.5_lrapa'], clean['wind_speed_mph'])
r_partial, p_partial = stats.pearsonr(pm_resid, wind_resid)

print('Holiday Farm Fire 2020 — PM2.5 vs wind speed')
print(f'  Raw correlation:              r = {r_raw:+.3f}  p = {p_raw:.4f}')
print(f'  Partial (controlling pressure): r = {r_partial:+.3f}  p = {p_partial:.4f}')
print()
if abs(r_partial) < abs(r_raw) * 0.5:
    print('  → Correlation largely explained by shared pressure confound')
elif p_partial < 0.05:
    print('  → Residual association persists after removing pressure effect')
else:
    print('  → No significant residual association after pressure control')

# Scatter of residuals
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (x, y, xlabel, ylabel, title) in zip(axes, [
    (clean['pm2.5_lrapa'], clean['wind_speed_mph'],
     'PM2.5 (µg/m³)', 'Wind speed (mph)', f'Raw  r={r_raw:.3f}'),
    (pm_resid, wind_resid,
     'PM2.5 residual (pressure removed)', 'Wind residual (pressure removed)',
     f'Partial  r={r_partial:.3f}'),
]):
    ax.scatter(x, y, s=6, alpha=0.3, color=COLOR_2020)
    m, b, *_ = stats.linregress(x, y)
    xr = np.linspace(np.nanmin(x), np.nanmax(x), 100)
    ax.plot(xr, m*xr+b, 'k--', linewidth=1.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

plt.suptitle('PM2.5 vs Wind Speed: raw vs pressure-controlled\n(Holiday Farm Fire 2020)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/fig_bl_partial_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

In [ ]:
smoke_w = df2020[df2020['is_smoke']]['wind_speed_mph'].dropna()
clean_w = df2020[~df2020['is_smoke']]['wind_speed_mph'].dropna()
t_w, p_w = stats.ttest_ind(smoke_w, clean_w, equal_var=False)
d_w = (smoke_w.mean()-clean_w.mean()) / np.sqrt((smoke_w.std()**2+clean_w.std()**2)/2)

smoke_t = df2020[df2020['is_smoke']]['temperature_f'].dropna()
clean_t = df2020[~df2020['is_smoke']]['temperature_f'].dropna()
t_t, p_t = stats.ttest_ind(smoke_t, clean_t, equal_var=False)

print('=' * 60)
print('BOUNDARY LAYER — Holiday Farm Fire 2020 summary')
print('=' * 60)
print(f'Wind speed (hourly):')
print(f'  Clean: {clean_w.mean():.1f} mph   Smoke: {smoke_w.mean():.1f} mph   '
      f'Δ={smoke_w.mean()-clean_w.mean():+.1f}   p={p_w:.4f}   d={d_w:.2f}')
print(f'Temperature (hourly):')
print(f'  Clean: {clean_t.mean():.1f}°F   Smoke: {smoke_t.mean():.1f}°F   '
      f'Δ={smoke_t.mean()-clean_t.mean():+.1f}°F   p={p_t:.4f}')
print(f'\nLagged PM2.5 → wind correlations: see lag plot above')
print(f'Partial r (pressure removed): see section 6')